# Setup

In [1]:
# base
import os
import sys
import warnings
import logging
import pickle
from pathlib import Path

# data manipulation
import pickle
import pandas as pd
import itables
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import scipy

# single cell
import scanpy as sc

itables.init_notebook_mode(connected=True)  # Use connected=False for offline use
warnings.simplefilter("ignore", FutureWarning)
warnings.simplefilter("ignore", UserWarning)
warnings.simplefilter("ignore", RuntimeWarning)
warnings.simplefilter("ignore", pd.errors.DtypeWarning)
warnings.simplefilter("ignore", pd.errors.PerformanceWarning)
warnings.simplefilter("ignore", DeprecationWarning)
mlogger = logging.getLogger("matplotlib")
mlogger.setLevel(logging.ERROR)
mlogger = logging.getLogger("harmonypy")
mlogger.setLevel(logging.ERROR)

# custom
from single_cell.R import *
from single_cell.preprocess import *
from single_cell.plot import *
from single_cell.analysis import *
from spatial_seq.plot import *
from utils import *

from rpy2.robjects.conversion import localconverter

converter = get_converter()

# R_preload()
%load_ext rpy2.ipython

study = "Kohda2025"

CORES = 10
DATADIR = Path("../../../data")
REFDIR = Path("../../../references")
DOUBLETMETHODS = ["scDblFinder", "DoubletFinder", "doubletdetection", "scrublet"]

%matplotlib inline
mpl.rcdefaults()

/mnt/DATA/home/ethung/spatial_seq/.venv/lib/python3.12/site-packages/louvain/__init__.py:54: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound


# Access & Preprocess adata

In [36]:
study_name = "Kohda2025_download"
experiment_dir = Path(DATADIR / "counts" / study_name)

adata = sc.AnnData(
    pd.read_csv(
        experiment_dir / "GSM8491338_matrix_inflection_demulti_DBEC_WTA2.txt",
        sep="\t",
        index_col=0,
    ).T
)
adata.obs["doublet"] = adata.obs_names.str.contains("doublet")
adata = adata[~adata.obs["doublet"]]

In [ ]:
adata.obs["Identifier"] = (
    adata.obs_names.str.split("_", expand=True).to_frame()[0].tolist()
)

data = [
    ["A0301", "standard diet", "24 weeks", "not applicable"],
    ["A0302", "standard diet", "24 weeks", "not applicable"],
    ["A0303", "standard diet", "24 weeks", "not applicable"],
    ["A0304", "high-fat diet", "16 weeks", "8 weeks"],
    ["A0305", "high-fat diet", "16 weeks", "8 weeks"],
    ["A0307", "high-fat diet", "16 weeks", "8 weeks"],
    ["A0308", "high-fat diet", "24 weeks", "8 weeks"],
    ["A0309", "high-fat diet", "24 weeks", "8 weeks"],
    ["A0310", "high-fat diet", "24 weeks", "8 weeks"],
]

metadata = pd.DataFrame(
    data, columns=["Identifier", "Diet", "Age", "HFD Length"]
).set_index("Identifier")
metadata["Depot"] = "eWAT"
metadata["Gender"] = "Male"
metadata = metadata.to_dict()
for col in metadata:
    adata.obs[col] = adata.obs["Identifier"].map(metadata[col])

adata = adata[~(adata.obs["Identifier"] == "not-detected")]
adata

In [ ]:
NEW_NAMES = False

if NEW_NAMES is True:
    adata.obs_names = generate_barcodes(adata.shape[0])

annotation = "doublet_cleaned"
adata.write(DATADIR / "processed" / "single_cell" / study / f"{annotation}.h5ad")

In [2]:
annotation = "doublet_cleaned"
adata = sc.read_h5ad(
    DATADIR / "processed" / "single_cell" / study / f"{annotation}.h5ad"
)
adata

AnnData object with n_obs × n_vars = 11762 × 29417
    obs: 'doublet', 'Identifier', 'Diet', 'Age', 'HFD Length', 'Depot', 'Gender'